In [ ]:
%%pyspark
# ── Audit Gold 2: one report table per EA audit report, as materialized lake views (23 Nov 2026) ───────────────────
# Default lakehouse: LH_AUDIT_Gold2. Its dbo schema holds shortcuts to the Gold tables of PROC, Fin, SD (SO) and MD;
# every view below reads those shortcuts only (dbo.<table>), nothing outside this lakehouse.
# Why: SM_AUDIT ran out of memory reading the full Gold facts (G/L 119.7M rows) and reading values across from a second
# fact (R14 PO values, R05 paid amount, R09 on-time share). Each Fact_Rpt_..._MLV table holds:
#   - only the report's rows: its Is_Audit_ flag (the deck's fixed rules, built in Fin / SD / PROC Gold);
#   - the report's fact columns, the natural key, the date Dim_Date relates on, and the SK of every dimension the page
#     uses. No dimension attributes are copied in: the semantic model relates the shared dimension shortcuts
#     (Dim_Company_Code, Dim_Vendor, Dim_Material, Dim_PO_Header, ...) to each table, dimension -> fact only;
#   - the values of a second fact joined in here (R05, R09, R14), so the model has no cross-fact values or measures.
# The site value (company code / sales org / purchasing org) and time are page filters, not applied here.
# Same frame as NB_Gold2_OPER: drop (this cell), one CREATE ... _MLV1 cell per view, rename (Python cell near the end),
# then a check cell. Every view name ends in _MLV (house rule); the build name is ..._MLV1.
# Run after the Fin, SD and PROC Gold runs (incl. PROC's Fact_OTD and phase-2 cells), then refresh SM_AUDIT.
# Column lists: Audit/Audit_Report_Mapping_v8.xlsx (tab "Gold 2 (23 Nov)").

# ── 1. Check the sources, then drop every view (Python cell) ───────────────────────────────────────────────────────
# For each view in VIEWS:
#   - the build name (..._MLV1) and the final name are dropped with DROP MATERIALIZED LAKE VIEW (not DROP TABLE);
#   - the folder left under the final name by the last run's rename is a plain Delta table, which DROP MATERIALIZED
#     LAKE VIEW does not remove: that folder is deleted here. Only names in VIEWS are touched, and each holds rows this
#     notebook rebuilds. Other tables in the lakehouse (the shortcuts, fact_audit, ...) are never touched.
# Nothing is dropped when a source shortcut is missing, so the last run's tables stay readable.
# Every run is a full rebuild (no incremental refresh). A new view = its name in VIEWS + its CREATE cell (..._MLV1).
# The same list is repeated in the rename and check cells at the end, so they also run on their own: keep them equal.

VIEW_SCHEMA = 'dbo'
VIEWS = [
    'Fact_Rpt_Vendor_Pay_Commissions_MLV',
    'Fact_Rpt_Sales_Incoterms_MLV',
    'Fact_Rpt_Customer_Payments_MLV',
    'Fact_Rpt_Credit_Memos_MLV',
    'Fact_Rpt_IA_Vendor_Payments_MLV',
    'Fact_Rpt_PO_Release_Strategy_MLV',
    'Fact_Rpt_Non_PO_Invoices_MLV',
    'Fact_Rpt_PO_Schedule_Lines_MLV',
    'Fact_Rpt_Invoices_per_Period_MLV',
    'Fact_Rpt_AUC_Maintenance_MLV',
    'Fact_Rpt_PPV_MLV',
    'Fact_Rpt_Vendor_Payments_SM_MLV',
]

# The shortcuts the views read (PROC, Fin, SD Gold), all in this lakehouse's dbo schema.
SOURCES = [
    'Fact_GL_Accounting_Doc',
    'Fact_Billing_Doc_Item',
    'Fact_FI_Billing_Payment',
    'Fact_PO_Header',
    'Fact_PO_Delivery_Status',
    'Fact_PO_LineItem_Schedule',
    'Fact_OTD',
]

missing = [t for t in SOURCES if not spark.catalog.tableExists(f'{VIEW_SCHEMA}.{t}')]
if missing:
    raise RuntimeError(f'Shortcut(s) missing in {VIEW_SCHEMA}: {missing}. Add them to LH_AUDIT_Gold2; nothing was dropped.')

def drop_view(name):
    try:
        spark.sql(f'DROP MATERIALIZED LAKE VIEW IF EXISTS {VIEW_SCHEMA}.{name}')
    except Exception as e:     # the name is a plain table (the renamed folder), not a view: its folder goes below
        print(f'  {name}: not a materialized view ({str(e)[:100]})')

for view in VIEWS:
    for name in [view, view + '1']:
        drop_view(name)
    # The plain table left by the last rename. Its folder is read from the table itself (always in this lakehouse),
    # not from a source table as NB_Gold2_OPER does: the sources here are shortcuts to other lakehouses.
    if spark.catalog.tableExists(f'{VIEW_SCHEMA}.{view}'):
        path = spark.sql(f'DESCRIBE DETAIL {VIEW_SCHEMA}.{view}').collect()[0]['location'].rstrip('/')
        notebookutils.fs.rm(path, True)
        print(f'Deleted the folder {path} (left by the last rename)')
    print(f'Dropped (if they existed): {VIEW_SCHEMA}.{view}, {VIEW_SCHEMA}.{view}1')

In [ ]:
-- ── 04 Vendor Payments - Commissions and Rebates (deck V4 slide 10): Fact_Rpt_Vendor_Pay_Commissions_MLV ───────────
-- Rows: Fact_GL_Accounting_Doc where Is_Audit_Commission_Rebate. Key: Doc_Segment_SK (one row per G/L line).
-- Not built: Customer via Header › Description (needs Header_Vendor_SK / Header_Customer_SK on the G/L fact).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_Vendor_Pay_Commissions_MLV1

COMMENT "G/L lines in the Vendor Payments - Commissions and Rebates scope"

AS

SELECT
    Doc_Segment_SK, Company_Code_SK, Fiscal_Year, Document_Number, Item, Posting_Date, SAP_User_SK,
    GL_Account_SK, Clearing_Document, Execution_Status, Document_Type, Posting_Key, Amt_per_S_or_H,
    Local_Currency, Amt_DC_per_S_or_H, Currency_Key, Text, Assignment, Document_Header_Text, Net_Due_Date,
    Entry_Date, Document_Date, _CREATED_AT, _UPDATED_AT, _ROW_HASH, _PIPELINE_NAME, _PIPELINE_RUN_ID
FROM dbo.Fact_GL_Accounting_Doc
WHERE COALESCE(CAST(Is_Audit_Commission_Rebate AS BOOLEAN), FALSE)

In [ ]:
-- ── 05 Sales with Incoterms: Fact_Rpt_Sales_Incoterms_MLV ──────────────────────────────────────────────────────────
-- Rows: Fact_Billing_Doc_Item where Is_Audit_Sales_Incoterms. Key: Billing_Item_SK (one row per billing item).
-- Joined here: Paid_Billing_Amount from Fact_FI_Billing_Payment (Fin Gold, same key, one row per billing
-- item; left join, blank when not paid).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_Sales_Incoterms_MLV1

COMMENT "Billing items in the Sales with Incoterms scope, with the paid billing amount"

AS

SELECT
    b.Billing_Item_SK, b.Billing_Document_SK, b.Billing_Document, b.Billing_Item, b.Billing_Date,
    b.Company_Code_SK, b.Sales_Organization_SK, b.Plant_SK, b.Customer_Sales_SK, b.Customer_SK, b.Material_SK,
    b.Billing_Type, b.Order_Distribution_Channel, b.Payment_Terms, b.Incoterms, b.Payer, b.Payer_Name,
    b.Net_Value_Signed, b.Document_Currency, b.Sales_Document, b.Sales_Document_Type, b.Incoterms_Order_Header,
    b.Order_Bill_To_Party, b.Order_Bill_To_Party_Name, b.Order_Ship_To_Party, b.Order_Ship_To_Party_Name,
    b.Ship_To_Party, b.Ship_To_Party_Name, b.Actual_GI_Date, b.Delivery, b.Delivery_Date,
    b.Item_Delivery_Status, b.Created_By, b.Order_Payment_Terms_Header, b.Order_Payment_Terms_Item,
    b.Net_Value_Signed_USD_FIX, b.Order_Pricing_Date, b.Customer_PO_Number, b.Incoterms_Order_Item,
    b.Returns_Item, b._CREATED_AT, b._UPDATED_AT, b._ROW_HASH, b._PIPELINE_NAME, b._PIPELINE_RUN_ID,
    p.Paid_Billing_Amount
FROM dbo.Fact_Billing_Doc_Item AS b
LEFT JOIN dbo.Fact_FI_Billing_Payment AS p
    ON p.Billing_Item_SK = b.Billing_Item_SK
WHERE COALESCE(CAST(b.Is_Audit_Sales_Incoterms AS BOOLEAN), FALSE)

In [ ]:
-- ── 06 Customer Payments (slide 15): Fact_Rpt_Customer_Payments_MLV ────────────────────────────────────────────────
-- Rows: Fact_GL_Accounting_Doc where Is_Audit_Customer_Activity. Key: Doc_Segment_SK (one row per G/L line).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_Customer_Payments_MLV1

COMMENT "G/L lines in the Customer Payments scope"

AS

SELECT
    Doc_Segment_SK, Company_Code_SK, Fiscal_Year, Document_Number, Item, Posting_Date, Customer_SK,
    GL_Account_SK, Execution_Status, Document_Type, Document_Type_Desc, Posting_Key, Posting_Key_Desc,
    Amt_per_S_or_H, Local_Currency, Amt_DC_per_S_or_H, Currency_Key, Text, Assignment, _CREATED_AT,
    _UPDATED_AT, _ROW_HASH, _PIPELINE_NAME, _PIPELINE_RUN_ID
FROM dbo.Fact_GL_Accounting_Doc
WHERE COALESCE(CAST(Is_Audit_Customer_Activity AS BOOLEAN), FALSE)

In [ ]:
-- ── 07 Credit Memos: Fact_Rpt_Credit_Memos_MLV ─────────────────────────────────────────────────────────────────────
-- Rows: Fact_Billing_Doc_Item where Is_Audit_Credit_Memo. Key: Billing_Item_SK (one row per billing item).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_Credit_Memos_MLV1

COMMENT "Billing items in the Credit Memos scope"

AS

SELECT
    Billing_Item_SK, Billing_Document_SK, Billing_Document, Billing_Item, Billing_Date, Customer_SK,
    Material_SK, Billing_Type, Sales_Document, Payment_Terms, Payer, Payer_Name, Net_Value_Signed,
    Document_Currency, Sales_Document_Type, Incoterms, Posting_Status, Order_Reason, Order_Bill_To_Party,
    Order_Bill_To_Party_Name, Order_Ship_To_Party, Order_Ship_To_Party_Name, Sales_Created_By, Created_By,
    _CREATED_AT, _UPDATED_AT, _ROW_HASH, _PIPELINE_NAME, _PIPELINE_RUN_ID
FROM dbo.Fact_Billing_Doc_Item
WHERE COALESCE(CAST(Is_Audit_Credit_Memo AS BOOLEAN), FALSE)

In [ ]:
-- ── 08 IA Vendor Payments (slide 19): Fact_Rpt_IA_Vendor_Payments_MLV ──────────────────────────────────────────────
-- Rows: Fact_GL_Accounting_Doc where Is_Audit_Vendor_Payment. Key: Doc_Segment_SK (one row per G/L line).
-- Not built: Customer via Header › Description (needs Header_Vendor_SK / Header_Customer_SK on the G/L fact).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_IA_Vendor_Payments_MLV1

COMMENT "G/L lines in the IA Vendor Payments scope"

AS

SELECT
    Doc_Segment_SK, Company_Code_SK, Fiscal_Year, Document_Number, Item, Posting_Date, SAP_User_SK,
    GL_Account_SK, Clearing_Document, Execution_Status, Document_Type, Posting_Key, Amt_per_S_or_H,
    Local_Currency, Amt_DC_per_S_or_H, Currency_Key, Text, Assignment, Document_Header_Text, Net_Due_Date,
    Due_Date_Max_Discount, AP_AR_Clearing_Date, Entry_Date, Document_Date, Payment_Difference_Days,
    _CREATED_AT, _UPDATED_AT, _ROW_HASH, _PIPELINE_NAME, _PIPELINE_RUN_ID
FROM dbo.Fact_GL_Accounting_Doc
WHERE COALESCE(CAST(Is_Audit_Vendor_Payment AS BOOLEAN), FALSE)

In [ ]:
-- ── 09 PO Release Strategy: Fact_Rpt_PO_Release_Strategy_MLV ───────────────────────────────────────────────────────
-- Rows: every PO header (Fact_PO_Header). The deck has no fixed rule for this report, only the site value
-- (company code page filter). Release strategy texts come from Dim_PO_Header in the model. Key: PO_SK.
-- Joined here: OTD_On_Time_Share = share of the PO's receipt allocations delivered on time, the average of
-- Fact_OTD.OTD_Flag (1 = on time, -7 to +3 days) per PO_SK; same rule as the v6 measure [OTD · On-Time Share].
-- Open with Jeremy and Procurement team: OTD weighting (per receipt, as here, or by quantity). Blank = no receipt.
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_PO_Release_Strategy_MLV1

COMMENT "Every PO header with its on-time delivery share"

AS

WITH OTD_Per_PO AS
(
    SELECT PO_SK, AVG(CAST(OTD_Flag AS DOUBLE)) AS OTD_On_Time_Share
    FROM dbo.Fact_OTD
    GROUP BY PO_SK
)
SELECT
    h.PO_SK, h.Purchasing_Document, h.Created_On, h.Vendor_SK, h.Company_Code_SK, h.Purch_Org_SK,
    h.Purchasing_Group_SK, h.Execution_Status, h.Net_Order_Value, h.Received_Value_DC, h.Invoiced_Value_LC,
    h.Invoiced_Quantity, h.Created_By, h._CREATED_AT, h._UPDATED_AT, h._ROW_HASH, h._PIPELINE_NAME,
    h._PIPELINE_RUN_ID,
    o.OTD_On_Time_Share
FROM dbo.Fact_PO_Header AS h
LEFT JOIN OTD_Per_PO AS o
    ON o.PO_SK = h.PO_SK

In [ ]:
-- ── 10 Non-PO Invoices (slide 21): Fact_Rpt_Non_PO_Invoices_MLV ────────────────────────────────────────────────────
-- Rows: Fact_GL_Accounting_Doc where Is_Audit_Non_PO_Invoice. Key: Doc_Segment_SK (one row per G/L line).
-- Not built: Vendor via Header › ID and Description (needs Header_Vendor_SK / Header_Customer_SK on the G/L fact).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_Non_PO_Invoices_MLV1

COMMENT "G/L lines in the Non-PO Invoices scope"

AS

SELECT
    Doc_Segment_SK, Company_Code_SK, Fiscal_Year, Document_Number, Item, Posting_Date, GL_Account_SK,
    Fiscal_Period, Document_Date, Document_Type, Posting_Key, Amt_per_S_or_H, Local_Currency,
    Amt_DC_per_S_or_H, Header_Currency_Key, Baseline_Date, AP_AR_Clearing_Date, Clearing_Document, _CREATED_AT,
    _UPDATED_AT, _ROW_HASH, _PIPELINE_NAME, _PIPELINE_RUN_ID
FROM dbo.Fact_GL_Accounting_Doc
WHERE COALESCE(CAST(Is_Audit_Non_PO_Invoice AS BOOLEAN), FALSE)

In [ ]:
-- ── 11 PO Schedule Lines (goods in transit): Fact_Rpt_PO_Schedule_Lines_MLV ────────────────────────────────────────
-- Rows: Fact_PO_LineItem_Schedule where Is_Audit_Goods_In_Transit. Key: PO_Schedule_Status_SK (one row per PO schedule line).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_PO_Schedule_Lines_MLV1

COMMENT "PO schedule lines in the Goods in Transit scope"

AS

SELECT
    PO_Schedule_Status_SK, PO_SK, PO_Item_SK, Purchasing_Document, Item, Schedule_Line, Header_Created_On,
    Purch_Org_SK, Plant_SK, Material_SK, Vendor_SK, Purchasing_Group_SK, Execution_Status, Quantity_Base_UOM,
    Short_Text, MRP_Controller, Order_Status_Purchase, Order_Due_Date, Delivery_Reliability, Ordered_Value,
    Goods_Receipt_Date, Vendor_PO_Incoterms, Realized_Finish_Date, Planned_Date, Plant_Country_Of_Origin,
    Plant_Profit_Center, _CREATED_AT, _UPDATED_AT, _ROW_HASH, _PIPELINE_NAME, _PIPELINE_RUN_ID
FROM dbo.Fact_PO_LineItem_Schedule
WHERE COALESCE(CAST(Is_Audit_Goods_In_Transit AS BOOLEAN), FALSE)

In [ ]:
-- ── 12 Invoices per Period (slide 23): Fact_Rpt_Invoices_per_Period_MLV ────────────────────────────────────────────
-- Rows: Fact_GL_Accounting_Doc where Is_Audit_Duplicate_Payment. Key: Doc_Segment_SK (one row per G/L line).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_Invoices_per_Period_MLV1

COMMENT "G/L lines in the Invoices per Period scope"

AS

SELECT
    Doc_Segment_SK, Company_Code_SK, Fiscal_Year, Document_Number, Item, Posting_Date, Vendor_SK, SAP_User_SK,
    Invoices_Period_ID, Reference, Text, Execution_Status, Document_Type, Document_Type_Desc, Payment_Block,
    Payment_Block_Desc, Transaction_Code, Amt_per_S_or_H, Local_Currency, Amt_DC_per_S_or_H,
    Header_Currency_Key, Currency_Key, Posting_Key, Payment_Transaction, Purchasing_Document,
    Clearing_Document, Document_Date, Document_Header_Text, IR_Reference, IR_Document_Date, IR_Document_Type,
    IR_PD_Items_Related_To_Invoice, IR_Invoice_Document, IR_Transaction_Type, IR_Entry_Date, IR_Invoice_Status,
    Parked_By, IR_User_Name, _CREATED_AT, _UPDATED_AT, _ROW_HASH, _PIPELINE_NAME, _PIPELINE_RUN_ID
FROM dbo.Fact_GL_Accounting_Doc
WHERE COALESCE(CAST(Is_Audit_Duplicate_Payment AS BOOLEAN), FALSE)

In [ ]:
-- ── 13 AUC Maintenance (slide 27): Fact_Rpt_AUC_Maintenance_MLV ────────────────────────────────────────────────────
-- Rows: Fact_GL_Accounting_Doc where Is_Audit_Maintenance_Expense. Key: Doc_Segment_SK (one row per G/L line).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_AUC_Maintenance_MLV1

COMMENT "G/L lines in the AUC Maintenance scope"

AS

SELECT
    Doc_Segment_SK, Company_Code_SK, Fiscal_Year, Document_Number, Item, Posting_Date, GL_Account_SK,
    Vendor_SK, PO_SK, PO_Item_SK, Material_SK, Execution_Status, Document_Type, Posting_Key, Amt_per_S_or_H,
    Local_Currency, Amt_DC_per_S_or_H, Header_Currency_Key, Assignment, `Order`, Order_Desc,
    Document_Header_Text, Text, Cost_Center, Cost_Center_Name, _CREATED_AT, _UPDATED_AT, _ROW_HASH,
    _PIPELINE_NAME, _PIPELINE_RUN_ID
FROM dbo.Fact_GL_Accounting_Doc
WHERE COALESCE(CAST(Is_Audit_Maintenance_Expense AS BOOLEAN), FALSE)

In [ ]:
-- ── 14 Purchase Price Variances (slide 29): Fact_Rpt_PPV_MLV ───────────────────────────────────────────────────────
-- Rows: G/L lines where Is_Audit_PPV. Key: Doc_Segment_SK. Joined here (left joins, one row per G/L line kept):
--   - 9 PO header values from Fact_PO_Header on PO_SK, named PO_Header_<column>      (EA 'PD Header ›');
--   - 13 PO item values from Fact_PO_Delivery_Status on PO_Item_SK, named PO_Item_<column> (EA 'PD Item ›').
-- Both facts are unique on their key, so the joins add no rows. Header values repeat on each G/L line of the
-- same PO: totals stay off in the visual. Price_Unit and Qty_Conversion_Num stay in Dim_PO_Item.
-- Not built: 'Vendor via Header' (needs Header_Vendor_SK on the G/L fact).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_PPV_MLV1

COMMENT "G/L lines in the Purchase Price Variance scope, with their PO header and PO item values"

AS

SELECT
    g.Doc_Segment_SK, g.Company_Code_SK, g.Fiscal_Year, g.Document_Number, g.Item, g.Posting_Date, g.PO_SK,
    g.SAP_User_SK, g.GL_Account_SK, g.Vendor_SK, g.Material_SK, g.PO_Item_SK, g.Execution_Status,
    g.Document_Type, g.Posting_Key, g.Amt_per_S_or_H, g.Local_Currency, g.Amt_DC_per_S_or_H, g.Currency_Key,
    g.Text, g.Assignment, g.Document_Header_Text, g.Cost_Center, g.Profit_Center_Desc,
    g.Profit_Center_Hierarchy_Area, g.Profit_Center, g.Document_Date, g._CREATED_AT, g._UPDATED_AT,
    g._ROW_HASH, g._PIPELINE_NAME, g._PIPELINE_RUN_ID,
    h.Purchase_Value AS PO_Header_Purchase_Value,
    h.GR_Amount_LC AS PO_Header_GR_Amount_LC,
    h.Material_Value AS PO_Header_Material_Value,
    h.Net_Order_Value_USD_FIX AS PO_Header_Net_Order_Value_USD_FIX,
    h.Invoiced_Value_LC AS PO_Header_Invoiced_Value_LC,
    h.Received_Quantity_Base_UOM AS PO_Header_Received_Quantity_Base_UOM,
    h.Ordered_Quantity_Base_UOM AS PO_Header_Ordered_Quantity_Base_UOM,
    h.Quantity_Open_Active_Base_UOM AS PO_Header_Quantity_Open_Active_Base_UOM,
    h.Invoiced_Quantity_Base_UOM AS PO_Header_Invoiced_Quantity_Base_UOM,
    i.Net_Order_Value AS PO_Item_Net_Order_Value,
    i.GR_Amount_LC AS PO_Item_GR_Amount_LC,
    i.Material_Value AS PO_Item_Material_Value,
    i.Item_Value AS PO_Item_Item_Value,
    i.Invoiced_Quantity_Base_UOM AS PO_Item_Invoiced_Quantity_Base_UOM,
    i.Received_Quantity_Base_UOM AS PO_Item_Received_Quantity_Base_UOM,
    i.Ordered_Quantity_Base_UOM AS PO_Item_Ordered_Quantity_Base_UOM,
    i.Quantity_Open_Active AS PO_Item_Quantity_Open_Active,
    i.Material_Moving_Price AS PO_Item_Material_Moving_Price,
    i.Material_Price_Unit AS PO_Item_Material_Price_Unit,
    i.Material_Plant_Price_Per_Unit AS PO_Item_Material_Plant_Price_Per_Unit,
    i.Price_Per_Unit_LC AS PO_Item_Price_Per_Unit_LC,
    i.Price_Per_Unit_DC AS PO_Item_Price_Per_Unit_DC
FROM dbo.Fact_GL_Accounting_Doc AS g
LEFT JOIN dbo.Fact_PO_Header AS h
    ON h.PO_SK = g.PO_SK
LEFT JOIN dbo.Fact_PO_Delivery_Status AS i
    ON i.PO_Item_SK = g.PO_Item_SK
WHERE COALESCE(CAST(g.Is_Audit_PPV AS BOOLEAN), FALSE)

In [ ]:
-- ── 15 Vendor Payments SM: Fact_Rpt_Vendor_Payments_SM_MLV ─────────────────────────────────────────────────────────
-- Rows: Fact_GL_Accounting_Doc where Is_Audit_Compliance_Payment. Key: Doc_Segment_SK (one row per G/L line).
-- Not built: Customer via Header › Description, Vendor via Header › Description, Vendor via Header › ID, Vendor via Header › Vendor account group (needs Header_Vendor_SK / Header_Customer_SK on the G/L fact).
CREATE OR REPLACE MATERIALIZED LAKE VIEW dbo.Fact_Rpt_Vendor_Payments_SM_MLV1

COMMENT "G/L lines in the Vendor Payments SM scope"

AS

SELECT
    Doc_Segment_SK, Company_Code_SK, Fiscal_Year, Document_Number, Item, Posting_Date, SAP_User_SK,
    GL_Account_SK, PO_SK, PO_Item_SK, Clearing_Document, Execution_Status, Document_Type, Posting_Key,
    Amt_per_S_or_H, Local_Currency, Amt_DC_per_S_or_H, Currency_Key, Text, Document_Header_Text, Net_Due_Date,
    IR_Net_Due_Date, Invoice_With_Purchase_Order, AP_AR_Clearing_Date, Clearing_Date, Entry_Date,
    Document_Date, IR_Entry_Date, IR_Document_Date, Payment_Difference_Days, IR_Invoice_Document, _CREATED_AT,
    _UPDATED_AT, _ROW_HASH, _PIPELINE_NAME, _PIPELINE_RUN_ID
FROM dbo.Fact_GL_Accounting_Doc
WHERE COALESCE(CAST(Is_Audit_Compliance_Payment AS BOOLEAN), FALSE)

In [ ]:
%%pyspark
# ── Rename: every view's table folder to its final name (Python cell, run after all the CREATE cells) ────────
# Same cell as NB_Gold2_OPER (3 Nov 2026), with this notebook's VIEWS.
# Spark SQL stores a materialized lake view name in lower case (documented behaviour), so ALTER ... RENAME TO cannot
# give a view its capitals. This cell renames the Delta folder of each view instead, as the portal does by hand.
# final_name(): the words keep their capitals and stay joined by '_', and a 1 at the end is removed:
# Fact_Rpt_PPV_MLV1 becomes Fact_Rpt_PPV_MLV.
# Effect: each folder is a plain Delta table under the final name holding the rows built above. The view's own
# registration (..._mlv1: refresh schedule, lineage) points at the old folder, so it is dropped after the move. The
# tables are rebuilt by the next run of this notebook; they are not refreshed by themselves.
# A view that fails is reported and the others still run; the cell raises at the end if any failed.

import re

# own settings, so this cell also runs on its own (same list as the drop cell at the top)
VIEW_SCHEMA = 'dbo'
VIEWS = [
    'Fact_Rpt_Vendor_Pay_Commissions_MLV',
    'Fact_Rpt_Sales_Incoterms_MLV',
    'Fact_Rpt_Customer_Payments_MLV',
    'Fact_Rpt_Credit_Memos_MLV',
    'Fact_Rpt_IA_Vendor_Payments_MLV',
    'Fact_Rpt_PO_Release_Strategy_MLV',
    'Fact_Rpt_Non_PO_Invoices_MLV',
    'Fact_Rpt_PO_Schedule_Lines_MLV',
    'Fact_Rpt_Invoices_per_Period_MLV',
    'Fact_Rpt_AUC_Maintenance_MLV',
    'Fact_Rpt_PPV_MLV',
    'Fact_Rpt_Vendor_Payments_SM_MLV',
]


def final_name(name):
    words = '_'.join(word[:1].upper() + word[1:] for word in name.split('_'))
    return re.sub(r'1$', '', words)

def rename_view(view_final):
    view_build = view_final + '1'
    new_name = final_name(view_build)
    if new_name != view_final:
        raise ValueError(f'final_name({view_build}) gave {new_name}, expected {view_final}')

    src_path = spark.sql(
        f'DESCRIBE DETAIL {VIEW_SCHEMA}.{view_build}'
    ).collect()[0]['location'].rstrip('/')
    # the final folder goes next to the view's own folder, in this lakehouse (the sources are shortcuts, 23 Nov 2026)
    dst_path = f"{src_path.rsplit('/', 1)[0]}/{new_name}"
    if notebookutils.fs.exists(dst_path):
        raise ValueError(f'{dst_path} exists already: run the drop cell at the top first')

    rows_before = spark.read.format('delta').load(src_path).count()
    notebookutils.fs.mv(src_path, dst_path, True)
    rows_after = spark.read.format('delta').load(dst_path).count()
    if rows_before != rows_after:
        raise ValueError(f'row count changed in the move: {rows_before:,} before, {rows_after:,} after')
    print(f'Renamed {view_build} → {new_name}  ({rows_after:,} rows)')

    # the old registration points at a folder that is gone now: remove it (a failure here is not fatal)
    try:
        spark.sql(f'DROP MATERIALIZED LAKE VIEW IF EXISTS {VIEW_SCHEMA}.{view_build}')
    except Exception as e:
        print(f'  Could not remove the registration {VIEW_SCHEMA}.{view_build} ({str(e)[:150]}): drop it by hand')

failed = {}
for view in VIEWS:
    try:
        rename_view(view)
    except Exception as e:
        failed[view] = str(e)[:200]
        print(f'FAILED {view}: {failed[view]}')

print('Refresh the lakehouse explorer to see the tables.')
if failed:
    raise RuntimeError(f'{len(failed)} of {len(VIEWS)} view(s) not renamed: {sorted(failed)}')

In [ ]:
%%pyspark
# ── Checks: rows, keys and joined values per report table (Python cell, warn-only, printed) ────────────────────────
# For each table built above:
#   - rows = rows in the report scope (source rows with the flag; R09: every PO header), so a join added no rows;
#   - no duplicate key;
#   - R05 / R09 / R14: share of rows where the joined value was found. Low = PO_SK / PO_Item_SK missing on G/L lines,
#     or no payment / receipt yet. Warn below 50 %.
# Each source is read once (one pass counts every flag). Nothing is written or changed; set RUN_CHECKS = False to skip.
from pyspark.sql import functions as F

RUN_CHECKS = globals().get('RUN_CHECKS', True)
VIEW_SCHEMA = 'dbo'
PPV_JOIN = [
    'PO_Header_Purchase_Value', 'PO_Header_GR_Amount_LC', 'PO_Header_Material_Value',
    'PO_Header_Net_Order_Value_USD_FIX', 'PO_Header_Invoiced_Value_LC', 'PO_Header_Received_Quantity_Base_UOM',
    'PO_Header_Ordered_Quantity_Base_UOM', 'PO_Header_Quantity_Open_Active_Base_UOM', 'PO_Header_Invoiced_Quantity_Base_UOM',
    'PO_Item_Net_Order_Value', 'PO_Item_GR_Amount_LC', 'PO_Item_Material_Value',
    'PO_Item_Item_Value', 'PO_Item_Invoiced_Quantity_Base_UOM', 'PO_Item_Received_Quantity_Base_UOM',
    'PO_Item_Ordered_Quantity_Base_UOM', 'PO_Item_Quantity_Open_Active', 'PO_Item_Material_Moving_Price',
    'PO_Item_Material_Price_Unit', 'PO_Item_Material_Plant_Price_Per_Unit', 'PO_Item_Price_Per_Unit_LC',
    'PO_Item_Price_Per_Unit_DC',
]
# table: (key, source, flag or None = every row, joined columns)
CHECKS = {
    'Fact_Rpt_Vendor_Pay_Commissions_MLV': ('Doc_Segment_SK', 'Fact_GL_Accounting_Doc', 'Is_Audit_Commission_Rebate', []),
    'Fact_Rpt_Sales_Incoterms_MLV': ('Billing_Item_SK', 'Fact_Billing_Doc_Item', 'Is_Audit_Sales_Incoterms', ['Paid_Billing_Amount']),
    'Fact_Rpt_Customer_Payments_MLV': ('Doc_Segment_SK', 'Fact_GL_Accounting_Doc', 'Is_Audit_Customer_Activity', []),
    'Fact_Rpt_Credit_Memos_MLV': ('Billing_Item_SK', 'Fact_Billing_Doc_Item', 'Is_Audit_Credit_Memo', []),
    'Fact_Rpt_IA_Vendor_Payments_MLV': ('Doc_Segment_SK', 'Fact_GL_Accounting_Doc', 'Is_Audit_Vendor_Payment', []),
    'Fact_Rpt_PO_Release_Strategy_MLV': ('PO_SK', 'Fact_PO_Header', None, ['OTD_On_Time_Share']),
    'Fact_Rpt_Non_PO_Invoices_MLV': ('Doc_Segment_SK', 'Fact_GL_Accounting_Doc', 'Is_Audit_Non_PO_Invoice', []),
    'Fact_Rpt_PO_Schedule_Lines_MLV': ('PO_Schedule_Status_SK', 'Fact_PO_LineItem_Schedule', 'Is_Audit_Goods_In_Transit', []),
    'Fact_Rpt_Invoices_per_Period_MLV': ('Doc_Segment_SK', 'Fact_GL_Accounting_Doc', 'Is_Audit_Duplicate_Payment', []),
    'Fact_Rpt_AUC_Maintenance_MLV': ('Doc_Segment_SK', 'Fact_GL_Accounting_Doc', 'Is_Audit_Maintenance_Expense', []),
    'Fact_Rpt_PPV_MLV': ('Doc_Segment_SK', 'Fact_GL_Accounting_Doc', 'Is_Audit_PPV', PPV_JOIN),
    'Fact_Rpt_Vendor_Payments_SM_MLV': ('Doc_Segment_SK', 'Fact_GL_Accounting_Doc', 'Is_Audit_Compliance_Payment', []),
}

def _flag(c):
    return F.coalesce(F.col(c).cast('boolean'), F.lit(False))

warnings = []
if RUN_CHECKS:
    scope = {}
    for src in sorted({s for _, s, _, _ in CHECKS.values()}):
        flags = sorted({f for _, s, f, _ in CHECKS.values() if s == src and f})
        aggs = [F.count(F.lit(1)).alias('_all')] + [F.sum(_flag(f).cast('long')).alias(f) for f in flags]
        row = spark.table(f'{VIEW_SCHEMA}.{src}').agg(*aggs).collect()[0]
        scope.update({(src, f): row[f] or 0 for f in flags})
        scope[(src, None)] = row['_all']

    for table, (key, src, flag, joined) in CHECKS.items():
        df = spark.table(f'{VIEW_SCHEMA}.{table}')
        aggs = [F.count(F.lit(1)).alias('n'), F.countDistinct(key).alias('keys')]
        if joined:
            aggs.append(F.sum(F.when(F.coalesce(*[F.col(c).cast('string') for c in joined]).isNotNull(), 1)
                              .otherwise(0)).alias('found'))
        r = df.agg(*aggs).collect()[0]
        n, keys, expected = r['n'], r['keys'], scope[(src, flag)]
        msg = f'{table}: {n:,} rows ({expected:,} in scope), {keys:,} distinct {key}'
        bad = []
        if n != expected:
            bad.append('rows differ from the report scope')
        if keys != n:
            bad.append(f'{n - keys:,} duplicate key(s)')
        if joined and n:
            pct = round(100.0 * (r['found'] or 0) / n, 1)
            msg += f', joined values found on {pct} %'
            if pct < 50:
                bad.append(f'joined values found on {pct} % of rows only')
        print(('⚠ ' if bad else '✓ ') + msg + (f'  ← {"; ".join(bad)}' if bad else ''))
        if bad:
            warnings.append(f'{table}: {"; ".join(bad)}')

print(f'\n{len(warnings)} warning(s)' + (': ' + ' | '.join(warnings) if warnings else '') if RUN_CHECKS else 'Checks skipped')

In [ ]:
# ── End of run: stop the session to release capacity (15 Nov 2026) ──────────────────────────────────────
# Same rules as end_session() in NB_Utils_Common, which this notebook does not load: skipped in a pipeline run
# (Fabric stops the session itself) and in a reference run; set END_SESSION = False in a cell above to keep it.
if globals().get('END_SESSION', True):
    try:
        _ctx = notebookutils.runtime.context
        if not (_ctx.get('isForPipeline') or _ctx.get('isReferenceRun')):
            print('Session stopped to release capacity (END_SESSION = False keeps it)')
            notebookutils.session.stop()
    except Exception as e:
        if 'STOP_SESSION_REQUEST' in str(e):
            raise    # Fabric stops the session through this marker exception: it must reach the runtime (18 Nov 2026)
        print(f'Session not stopped ({str(e)[:120]})')